In [45]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install gymnasium
!pip install stable-baselines3[extra]

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.2/187.2 kB 688.5 kB/s eta 0:00:00


In [ ]:
from stable_baselines3 import DQN
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.vec_env import DummyVecEnv
import pandas as pd

# Load flight and gate data
flights_df = pd.read_csv("/content/drive/MyDrive/FIT5126 Research/simulated_flight_gate_assignment.csv")

# Generate dummy gate dataframe
unique_gates = flights_df['Assigned_Gate'].dropna().unique()
gates_df = pd.DataFrame({
    'Gate_ID': unique_gates,
    'Is_Bridge': [1 if g.startswith('G') else 0 for g in unique_gates]
})

# Ensure datetime columns are parsed properly
flights_df["Actual_ARR"] = pd.to_datetime(flights_df["Actual_ARR"])
flights_df["Scheduled_DEP"] = pd.to_datetime(flights_df["Scheduled_DEP"])


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=

In [ ]:
import gymnasium as gym
from gymnasium import spaces
import numpy as np
import pandas as pd
from datetime import timedelta

class RealTimeGateEnv(gym.Env):
    """
    Environment for real-time gate reallocation based on delays.
    Each step processes a flight; the agent decides to reassign it or not.
    """
    def __init__(self, flights_df, gates_df, buffer_min=20):
        super().__init__()
        self.flights_df = flights_df.copy()
        self.gates_df = gates_df.copy()
        self.buffer_min = buffer_min
        self.num_gates = len(gates_df)
        self.num_flights = len(flights_df)

        self.action_space = spaces.Discrete(self.num_gates + 1)  # 0~n-1 = reassign; n = keep original
        self.observation_space = spaces.Dict({
            "flight_features": spaces.Box(low=0, high=1, shape=(6,), dtype=np.float32),
            "gate_mask": spaces.MultiBinary(self.num_gates)
        })

        self.remote_gate_map = self._build_remote_gate_map()
        self.current_idx = 0
        self.stand_usage = {g: [] for g in gates_df["Gate_ID"]}

    def _build_remote_gate_map(self):
        remote_stands = [f"R{20+i:02d}" for i in range(self.num_gates)]
        return dict(zip(self.gates_df[self.gates_df["Is_Bridge"] == 0]["Gate_ID"], remote_stands))

    def reset(self, *, seed=None, options=None):
        super().reset(seed=seed)
        self.current_idx = 0
        self.stand_usage = {g: [] for g in self.gates_df["Gate_ID"]}
        return self._get_obs(), {}

    def _get_obs(self):
        if self.current_idx >= self.num_flights:
            return {
                "flight_features": np.zeros(6, dtype=np.float32),
                "gate_mask": np.zeros(self.num_gates, dtype=np.int8)
            }

        row = self.flights_df.iloc[self.current_idx]
        arr = row["Actual_ARR"] - timedelta(minutes=self.buffer_min)
        dep = row["Scheduled_DEP"] + timedelta(minutes=self.buffer_min)

        gate_mask = []
        for _, gate_row in self.gates_df.iterrows():
            gate_id = gate_row["Gate_ID"]
            usage = self.stand_usage[gate_id]
            conflict = any((arr < end and dep > start) for start, end in usage)
            gate_mask.append(0 if conflict else 1)

        arr_min = row["Actual_ARR"].hour * 60 + row["Actual_ARR"].minute
        dep_min = row["Scheduled_DEP"].hour * 60 + row["Scheduled_DEP"].minute
        features = np.array([
            arr_min / 1440,
            dep_min / 1440,
            row["Delay_MIN"] / 180,
            row["Is_International"],
            int(row["Aircraft_Type"] in ["A330", "B777"]),
            0.0 if pd.isna(row["Assigned_Gate"]) else int(row["Assigned_Gate"][1:]) / self.num_gates
        ], dtype=np.float32)

        return {
            "flight_features": features,
            "gate_mask": np.array(gate_mask, dtype=np.int8)
        }

    def step(self, action):
        info = {}
        if self.current_idx >= self.num_flights:
            return self._get_obs(), 0.0, True, True, info

        row = self.flights_df.iloc[self.current_idx]
        arr = row["Actual_ARR"] - timedelta(minutes=self.buffer_min)
        dep = row["Scheduled_DEP"] + timedelta(minutes=self.buffer_min)

        assigned_gate = row["Assigned_Gate"]
        gate_id = assigned_gate if action == self.num_gates else self.gates_df.iloc[action]["Gate_ID"]

        usage = self.stand_usage.get(gate_id, [])
        conflict = any((arr < end and dep > start) for start, end in usage)

        if conflict or pd.isna(gate_id):
            reward = -5
        else:
            reward = 1 if gate_id == assigned_gate else 0.5
            self.stand_usage[gate_id].append((arr, dep))

        self.current_idx += 1
        terminated = self.current_idx >= self.num_flights
        truncated = False
        return self._get_obs(), reward, terminated, truncated, info

    def render(self):
        print(f"Flight {self.current_idx}")


In [ ]:
# Create and wrap environment
env = RealTimeGateEnv(flights_df, gates_df)
vec_env = DummyVecEnv([lambda: env])  # Wrap environment for DQN compatibility

In [ ]:
# Initialize and train DQN model
model = DQN("MultiInputPolicy", vec_env, verbose=1, learning_rate=1e-3, buffer_size=5000)
model.learn(total_timesteps=100000)


Using cpu device
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.962    |
| time/               |          |
|    episodes         | 4        |
|    fps              | 285      |
|    time_elapsed     | 1        |
|    total_timesteps  | 400      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 0.933    |
|    n_updates        | 74       |
----------------------------------
----------------------------------
| rollout/            |          |
|    exploration_rate | 0.924    |
| time/               |          |
|    episodes         | 8        |
|    fps              | 279      |
|    time_elapsed     | 2        |
|    total_timesteps  | 800      |
| train/              |          |
|    learning_rate    | 0.001    |
|    loss             | 1.29     |
|    n_updates        | 174      |
----------------------------------
----------------------------------
| rollout/            |          |
|  

In [ ]:
# Re-run environment with trained model to collect reassignment results
obs, _ = env.reset()
total_reward = 0
reassigned_flights = []

while True:
    action, _ = model.predict(obs)
    obs, reward, terminated, truncated, _ = env.step(action)
    total_reward += reward

    if env.current_idx <= env.num_flights:
        flight_row = env.flights_df.iloc[env.current_idx - 1]
        assigned_gate = (
            flight_row["Assigned_Gate"]
            if action == env.num_gates
            else env.gates_df.iloc[action]["Gate_ID"]
        )
        stand_id = (
            assigned_gate
            if assigned_gate in env.stand_usage
            else env.remote_gate_map.get(assigned_gate, None)
        )

        reassigned_flights.append({
            "Flight_ID": flight_row["Flight_ID"],
            "Assigned_Gate_Reassigned": assigned_gate,
            "Assigned_Stand_Reassigned": stand_id,
            "Reward": reward
        })

    if terminated or truncated:
        break

# Save reassignment info as DataFrame
reassigned_df = pd.DataFrame(reassigned_flights)
print(f"✅ Total reward from real-time reallocation: {total_reward}")


✅ Total reward from real-time reallocation: 42.5


In [ ]:
import pandas as pd

# 1. Merge reassignment results and rewards into the original flights dataframe
full_df = flights_df.copy()
full_df = full_df.merge(
    reassigned_df[['Flight_ID', 'Assigned_Gate_Reassigned', 'Assigned_Stand_Reassigned', 'Reward']],
    on='Flight_ID',
    how='left'
)

# 2. Check whether reassignment occurred (either gate or stand changed)
def was_reassigned(row):
    return int(  # convert to 0 or 1
        row['Assigned_Gate'] != row['Assigned_Gate_Reassigned'] or
        row['Assigned_Stand'] != row['Assigned_Stand_Reassigned']
    )

full_df['Reassigned'] = full_df.apply(was_reassigned, axis=1)

# 3. Summary statistics
total_flights = len(full_df)
total_reassigned = full_df['Reassigned'].sum()
reassignment_rate = total_reassigned / total_flights * 100 if total_flights else 0

print(f"[DQN] Total Flights: {total_flights}")
print(f"[DQN] Reassigned Flights: {total_reassigned}")
print(f"[DQN] Reassignment Rate (All Flights): {reassignment_rate:.2f}%")

# 4. Preview a sample of the results
print("\n[DQN] Sample of all flights with reassignment and reward info:")
print(full_df[['Flight_ID', 'Assigned_Gate', 'Assigned_Gate_Reassigned', 'Reassigned', 'Reward']].head())

# 5. Export full results to CSV
full_df.to_csv("all_flight_reassignment_analysis_dqn.csv", index=False)
print("✅ CSV file saved as 'all_flight_reassignment_analysis_dqn.csv'")


[DQN] Total Flights: 100
[DQN] Reassigned Flights: 74
[DQN] Reassignment Rate (All Flights): 74.00%

[DQN] Sample of all flights with reassignment and reward info:
  Flight_ID Assigned_Gate Assigned_Gate_Reassigned  Reassigned  Reward
0    MU9392           G08                      G08           1     1.0
1    HU4157           G01                      G01           0     1.0
2    HU4420           G02                      G02           1     1.0
3    ZH5380           G03                      G03           1     1.0
4    ZH3491           G04                      G24           1     0.5
✅ CSV file saved as 'all_flight_reassignment_analysis_dqn.csv'


/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:151: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)


In [46]:
def evaluate_model(model, env, num_runs=10):
    total_rewards = []
    all_reassigned = []

    for run in range(num_runs):
        obs, _ = env.reset()
        run_reward = 0
        reassigned_flights = []

        while True:
            action, _ = model.predict(obs)
            obs, reward, terminated, truncated, _ = env.step(action)
            run_reward += reward

            if env.current_idx <= env.num_flights:
                flight_row = env.flights_df.iloc[env.current_idx - 1]
                assigned_gate = (
                    flight_row["Assigned_Gate"]
                    if action == env.num_gates
                    else env.gates_df.iloc[action]["Gate_ID"]
                )
                stand_id = (
                    assigned_gate
                    if assigned_gate in env.stand_usage
                    else env.remote_gate_map.get(assigned_gate, None)
                )

                reassigned_flights.append({
                    "Run": run + 1,
                    "Flight_ID": flight_row["Flight_ID"],
                    "Assigned_Gate_Reassigned": assigned_gate,
                    "Assigned_Stand_Reassigned": stand_id,
                    "Reward": reward
                })

            if terminated or truncated:
                break

        total_rewards.append(run_reward)
        all_reassigned.extend(reassigned_flights)

        print(f"Run {run+1}: Reward = {run_reward:.2f}")

    avg_reward = np.mean(total_rewards)
    print(f"\n✅ Average reward over {num_runs} runs: {avg_reward:.2f}")

    reassigned_df = pd.DataFrame(all_reassigned)
    return reassigned_df, total_rewards

# For example, repeatedly evaluate the PPO model 10 times
reassigned_df, rewards = evaluate_model(model, env, num_runs=10)


Run 1: Reward = 43.50
Run 2: Reward = 50.50
Run 3: Reward = 44.00
Run 4: Reward = 45.00
Run 5: Reward = 47.00
Run 6: Reward = 44.00
Run 7: Reward = 50.00
Run 8: Reward = 41.00
Run 9: Reward = 22.00
Run 10: Reward = 50.50

✅ Average reward over 10 runs: 43.75
